In [55]:
from sdv.evaluation.single_table import evaluate_quality, run_diagnostic
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
import json
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import pandas as pd

# Load Data

In [2]:
real_train = pd.read_csv('../data/processed/v4/real_train_data.csv')
real_test = pd.read_csv('../data/processed/v4/real_test_data.csv')

# Load Models

In [3]:
import cloudpickle
import torch
import io

def load_synthesizer_cpu(filepath):
    with open(filepath, "rb") as f:
        buffer = io.BytesIO(f.read())
    
    # Monkey-patch torch.load to force CPU mapping
    original_torch_load = torch.load
    torch.load = lambda f, **kwargs: original_torch_load(
        f, map_location=torch.device("cpu"), **kwargs
    )
    
    try:
        synthesizer = cloudpickle.load(buffer)
    finally:
        torch.load = original_torch_load  # always restore original
    
    return synthesizer

# Load all three
v2_good_generator     = load_synthesizer_cpu("../models/v2/model_good.pkl")
v2_poor_generator     = load_synthesizer_cpu("../models/v2/model_poor.pkl")
v2_standard_generator = load_synthesizer_cpu("../models/v2/model_standard.pkl")

v2_models = [v2_good_generator, v2_poor_generator, v2_standard_generator]

/Users/luisejdm/Documents/ITESO/8vo Semestre/Deep Learning/Proyecto2_Deep_Learning-/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator BayesianGaussianMixture from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/luisejdm/Documents/ITESO/8vo Semestre/Deep Learning/Proyecto2_Deep_Learning-/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator BayesianGaussianMixture from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/luisejdm/Documents/ITESO/8vo Seme

In [4]:
v4_good_generator = CTGANSynthesizer.load("../models/v4/synth_good.pkl")
v4_poor_generator = CTGANSynthesizer.load("../models/v4/synth_poor.pkl")
v4_standard_generator = CTGANSynthesizer.load("../models/v4/synth_standard.pkl")

v4_models = [v4_good_generator, v4_poor_generator, v4_standard_generator]

/Users/luisejdm/Documents/ITESO/8vo Semestre/Deep Learning/Proyecto2_Deep_Learning-/venv/lib/python3.13/site-packages/sdv/_utils.py:503: FutureWarning: The 'load' function will be deprecated in future versions of SDV. Please use 'utils.load_synthesizer' instead.
  warnings.warn(
/Users/luisejdm/Documents/ITESO/8vo Semestre/Deep Learning/Proyecto2_Deep_Learning-/venv/lib/python3.13/site-packages/sdv/_utils.py:503: FutureWarning: The 'load' function will be deprecated in future versions of SDV. Please use 'utils.load_synthesizer' instead.
  warnings.warn(
/Users/luisejdm/Documents/ITESO/8vo Semestre/Deep Learning/Proyecto2_Deep_Learning-/venv/lib/python3.13/site-packages/sdv/_utils.py:503: FutureWarning: The 'load' function will be deprecated in future versions of SDV. Please use 'utils.load_synthesizer' instead.
  warnings.warn(


In [30]:
v2_good_metadata = Metadata.load_from_dict(json.load(open("../metadata/v2/good_metadata.json")))
v2_poor_metadata = Metadata.load_from_dict(json.load(open("../metadata/v2/poor_metadata.json")))
v2_standard_metadata = Metadata.load_from_dict(json.load(open("../metadata/v2/standard_metadata.json")))

v2_good_metadata.remove_column('Type_of_Loan')
v2_poor_metadata.remove_column('Type_of_Loan')
v2_standard_metadata.remove_column('Type_of_Loan')

/Users/luisejdm/Documents/ITESO/8vo Semestre/Deep Learning/Proyecto2_Deep_Learning-/venv/lib/python3.13/site-packages/sdv/metadata/metadata.py:205: UserWarning: No table name was provided to metadata containing only one table. Assigning name: table
  warnings.warn(


In [31]:
v4_good_metadata = Metadata.load_from_dict(json.load(open("../metadata/v4/train_good_metadata.json")))
v4_poor_metadata = Metadata.load_from_dict(json.load(open("../metadata/v4/train_poor_metadata.json")))
v4_standard_metadata = Metadata.load_from_dict(json.load(open("../metadata/v4/train_standard_metadata.json")))

In [32]:
v2_metadata = [v2_good_metadata, v2_poor_metadata, v2_standard_metadata]
v4_metadata = [v4_good_metadata, v4_poor_metadata, v4_standard_metadata]

In [33]:
v2_good_metadata.to_dict()['tables']['table']['columns'] == v4_good_metadata.to_dict()['tables']['table']['columns']

True

# Sample some data

In [34]:
n = len(real_train)
n

72155

In [39]:
v2_data = [model.sample(n) for model in v2_models]
v4_data = [model.sample(n) for model in v4_models]

v4_columns = v4_data[0].columns
v2_data = [df[v4_columns] for df in v2_data]

# Evaluate quality (from SDV)


In [40]:
real_goods = real_train[real_train['Credit_Score'] == 'Good']
real_poor = real_train[real_train['Credit_Score'] == 'Poor']
real_standard = real_train[real_train['Credit_Score'] == 'Standard']

In [41]:
order = ["GOOD", "POOR", "STANDARD"]

for i, (data, metadata) in enumerate(zip(v2_data, v2_metadata)):
    print(f"==========Evaluating {order[i]} v2 model...==========")
    quality_report = evaluate_quality(
        real_data=real_train,
        synthetic_data=data,
        metadata=metadata
    )
    print(quality_report)

for i, (data, metadata) in enumerate(zip(v4_data, v4_metadata)):
    print(f"==========Evaluating {order[i]} model...==========")
    quality_report = evaluate_quality(
        real_data=real_train,
        synthetic_data=data,
        metadata=metadata
    )
    print(quality_report)

==========Evaluating GOOD v2 model...==========
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 99.30it/s]| 
Column Shapes Score: 68.98%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:01<00:00, 175.45it/s]|
Column Pair Trends Score: 63.8%

Overall Score (Average): 66.39%

==========Evaluating POOR v2 model...==========
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 83.81it/s]|
Column Shapes Score: 75.95%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:01<00:00, 180.04it/s]|
Column Pair Trends Score: 72.17%

Overall Score (Average): 74.06%

==========Evaluating STANDARD v2 model...==========
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 63.72it/s]|
Column Shapes Score: 83.91%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:01<00:00, 180.36it/s]|
Column Pair Trends Score: 80.23%

Overall Score (Average): 82.07%

=========

# Statistical tests

In [58]:
def evaluate_synthetic_data(real_df, synthetic_df, categorical_cols=None, numeric_cols=None):
    """
    Evaluates synthetic tabular data against real data using:
    - KS test per numeric column
    - Chi-square test per categorical column
    - Correlation matrix comparison
    
    Parameters
    ----------
    real_df        : pd.DataFrame — real data
    synthetic_df   : pd.DataFrame — synthetic data
    categorical_cols : list of str, optional — inferred if not provided
    numeric_cols     : list of str, optional — inferred if not provided
    """

    # ── Infer column types if not provided ────────────────────────────────
    if numeric_cols is None:
        numeric_cols = real_df.select_dtypes(include="number").columns.tolist()
    if categorical_cols is None:
        categorical_cols = real_df.select_dtypes(include="object").columns.tolist()

    # ── 1. KS Test — numeric columns ──────────────────────────────────────
    print("=" * 60)
    print("KS TEST — NUMERIC COLUMNS")
    print("=" * 60)

    ks_results = []
    for col in numeric_cols:
        stat, p_value = stats.ks_2samp(
            real_df[col].dropna(),
            synthetic_df[col].dropna()
        )
        ks_results.append({
            "column"  : col,
            "ks_stat" : round(stat, 8),
            "p_value" : round(p_value, 8),
            "pass"    : p_value > 0.05  # fail = distributions significantly differ
        })

    ks_df = pd.DataFrame(ks_results).sort_values("p_value")
    print(ks_df.to_string(index=False))
    print(f"\nPassing columns: {ks_df['pass'].sum()} / {len(ks_df)}")

    # ── 2. Chi-Square Test — categorical columns ───────────────────────────
    print("\n" + "=" * 60)
    print("CHI-SQUARE TEST — CATEGORICAL COLUMNS")
    print("=" * 60)

    chi_results = []
    for col in categorical_cols:
        real_counts  = real_df[col].value_counts()
        synth_counts = synthetic_df[col].value_counts()

        # Align categories
        all_cats   = real_counts.index.union(synth_counts.index)
        real_freq  = real_counts.reindex(all_cats, fill_value=0)
        synth_freq = synth_counts.reindex(all_cats, fill_value=0)

        # Scale synthetic frequencies to match real total
        scale      = real_freq.sum() / synth_freq.sum()
        synth_freq = (synth_freq * scale).round()

        stat, p_value = stats.chisquare(f_obs=synth_freq, f_exp=real_freq)
        chi_results.append({
            "column"  : col,
            "chi_stat": round(stat, 4),
            "p_value" : round(p_value, 4),
            "pass"    : p_value > 0.05
        })

    chi_df = pd.DataFrame(chi_results).sort_values("p_value")
    print(chi_df.to_string(index=False))
    print(f"\nPassing columns: {chi_df['pass'].sum()} / {len(chi_df)}")

    # ── 3. Correlation Matrix Comparison ──────────────────────────────────
    print("\n" + "=" * 60)
    print("CORRELATION MATRIX COMPARISON")
    print("=" * 60)

    real_corr  = real_df[numeric_cols].corr()
    synth_corr = synthetic_df[numeric_cols].corr()
    corr_diff  = (real_corr - synth_corr).abs()

    mean_abs_diff = corr_diff.values[np.triu_indices_from(corr_diff.values, k=1)].mean()
    print(f"Mean absolute correlation difference: {mean_abs_diff:.4f}")
    print("(Values below 0.05 indicate strong correlation preservation)\n")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, matrix, title in zip(
        axes,
        [real_corr, synth_corr, corr_diff],
        ["Real Correlation", "Synthetic Correlation", "Absolute Difference"]
    ):
        im = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
        ax.set_title(title)
        ax.set_xticks(range(len(numeric_cols)))
        ax.set_yticks(range(len(numeric_cols)))
        ax.set_xticklabels(numeric_cols, rotation=90, fontsize=7)
        ax.set_yticklabels(numeric_cols, fontsize=7)
        plt.colorbar(im, ax=ax)

    plt.suptitle("Correlation Matrix Comparison — Real vs Synthetic", fontsize=13)
    plt.tight_layout()
    plt.show()

    return ks_df, chi_df, corr_diff

In [59]:
categorical_cols = [col for col in real_train.columns if real_train[col].dtype == "object"]
numeric_cols = [col for col in real_train.columns if real_train[col].dtype in ["int64", "float64"]]

In [126]:
def compare_models(real_list, v2_list, v4_list,
                   class_names=None, categorical_cols=None, numeric_cols=None):
    if class_names is None:
        class_names = ["good", "bad", "standard"]

    summary_rows = []

    for cls, real_df, v2_df, v4_df in zip(class_names, real_list, v2_list, v4_list):
        for model_label, syn_df in [("v2", v2_df), ("v4", v4_df)]:

            ks_pass_rate, mean_ks, chi_pass_rate, mean_corr_diff = evaluate_synthetic_data(
                real_df          = real_df,
                synthetic_df     = syn_df,
                categorical_cols = categorical_cols,
                numeric_cols     = numeric_cols
            )

            summary_rows.append({
                "class"         : cls,
                "model"         : model_label,
                "ks_pass_rate"  : round(ks_pass_rate, 4),
                "mean_ks_stat"  : round(mean_ks, 4),
                "chi_pass_rate" : round(chi_pass_rate, 4) if chi_pass_rate is not None else None,
                "mean_corr_diff": round(mean_corr_diff, 4)
            })

    summary_df = pd.DataFrame(summary_rows)

    # ── Helper ─────────────────────────────────────────────────────────────
    def get_winner(subset, col, minimize=True):
        vals = subset.set_index("model")[col].dropna()
        if vals.empty or vals.nunique() == 1:
            return "tie"
        idx = vals.idxmin() if minimize else vals.idxmax()
        return idx

    # ── Winner logic per class ─────────────────────────────────────────────
    winner_rows = []
    for cls in class_names:
        subset = summary_df[summary_df["class"] == cls]

        winner_ks   = get_winner(subset, "mean_ks_stat",  minimize=True)
        winner_chi  = get_winner(subset, "chi_pass_rate", minimize=False)
        winner_corr = get_winner(subset, "mean_corr_diff",minimize=True)


        winner_rows.append({
            "class"      : cls,
            "winner_ks"  : winner_ks,
            "winner_chi" : winner_chi,
            "winner_corr": winner_corr,
        })

    winners_df = pd.DataFrame(winner_rows).set_index("class")

    return summary_df, winners_df

In [127]:
# ── Usage ─────────────────────────────────────────────────────────────────
summary, winners = compare_models(
    real_list  = [real_goods, real_poor, real_standard],
    v2_list    = v2_data,
    v4_list    = v4_data,
    class_names= ["good", "poor", "standard"]
)

In [128]:
summary

,class,model,ks_pass_rate,mean_ks_stat,chi_pass_rate,mean_corr_diff
0,good,v2,0.0,0.1572,0.0,0.0739
1,good,v4,0.0,0.0825,0.0,0.1132
2,poor,v2,0.0,0.1529,0.0,0.0529
3,poor,v4,0.0,0.0671,0.0,0.0844
4,standard,v2,0.0,0.1437,0.0,0.0598
5,standard,v4,0.0,0.0465,0.0,0.0250


In [129]:
winners

,winner_ks,winner_chi,winner_corr
class,,,
good,v4,tie,v2
poor,v4,tie,v2
standard,v4,tie,v4


A pesar de que en el promedio de la diferencia en las matrices de correlación el modelo v2 gana en 2 de las 3 clases, se elige el modelo v4, pues es el que muestra el estadístico de ks promedio más en las 3 clases. Aunque ninguno de los datasets pasa las pruevas ks y chi cuadrada, elegimos el que mejores resultados estadísticos muestra y el que presentó una mayor estabilidad en el entrenamiento del modelo. 